# Day 069 — Exercise 3: Extract from Image

**What you'll build:** `extract_from_image(img_b64, schema_cls, describe_fn=None) -> dict` — the core extraction function combining prompt generation, LLM call, and JSON parsing.

**Why it matters:** This is the central function of the multimodal extraction pipeline. All other functions (`safe_extract`, `extract_pipeline`, `ImageExtractor.extract`) delegate to it. Getting the mock interface right ensures every downstream function is testable without Ollama.

In [ ]:
import re
import json
from pydantic import BaseModel

def build_extraction_prompt(schema_cls):
    schema_json = json.dumps(schema_cls.model_json_schema(), indent=2)
    return (
        'Extract structured data from this image and return ONLY valid JSON '
        'matching this schema exactly.\n\nSchema:\n' + schema_json +
        '\n\nReturn ONLY the JSON object, nothing else.'
    )

def strip_json_from_response(response):
    block = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
    if block:
        return block.group(1).strip()
    obj = re.search(r'\{[\s\S]*\}', response)
    if obj:
        return obj.group(0).strip()
    raise ValueError(f'No JSON found: {response[:200]!r}')

class ProductInfo(BaseModel):
    name:     str
    price:    float
    category: str = ''


## Task

Implement `extract_from_image(img_b64, schema_cls, describe_fn=None) -> dict`:

1. `prompt = build_extraction_prompt(schema_cls)`
2. If `describe_fn is not None`: `response = describe_fn(img_b64, prompt)`
3. Otherwise: call `ollama.chat(model='llava', messages=[{...}])`
4. `raw = strip_json_from_response(response)`
5. `return json.loads(raw)`

Return a plain dict — do not validate against the schema here.

## Your Implementation

In [ ]:
def extract_from_image(img_b64: str, schema_cls,
                       describe_fn=None) -> dict:
    """Send an image to a vision LLM and parse the response as JSON.

    Args:
        img_b64:     base64-encoded image string
        schema_cls:  Pydantic model class defining the extraction schema
        describe_fn: callable(img_b64, prompt) -> str for testing
    Returns:
        dict of extracted fields (not yet Pydantic-validated)
    Raises:
        ValueError if no JSON found in the response
        json.JSONDecodeError if the extracted string is malformed JSON
    """
    raise NotImplementedError


In [ ]:
def extract_from_image(img_b64: str, schema_cls,
                       describe_fn=None) -> dict:
    prompt = build_extraction_prompt(schema_cls)
    if describe_fn is not None:
        response = describe_fn(img_b64, prompt)
    else:
        import ollama
        resp = ollama.chat(
            model='llava',
            messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
        )
        response = resp['message']['content']
    raw = strip_json_from_response(response)
    return json.loads(raw)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    captured = {}
    def _mock(b64, p):
        captured['b64']    = b64
        captured['prompt'] = p
        return '{"name": "Headphones", "price": 49.99}'

    result = extract_from_image('test_b64==', ProductInfo, describe_fn=_mock)
    assert isinstance(result, dict), f"Expected dict, got {type(result)}"
    score += 1; print("\u2705 returns a dict")

    assert result == {'name': 'Headphones', 'price': 49.99}, (
        f"Unexpected result: {result}")
    score += 1; print("\u2705 dict matches mock JSON")

    assert captured['b64'] == 'test_b64==', "img_b64 should be passed to describe_fn"
    score += 1; print("\u2705 mock receives the img_b64 argument")

    # Prompt contains schema fields
    assert 'name' in captured['prompt'] and 'price' in captured['prompt'], (
        f"Schema fields missing from prompt: {captured['prompt'][:200]!r}")
    score += 1; print("\u2705 prompt contains schema field names")

    # Mock returning markdown block also works
    r2 = extract_from_image('b64', ProductInfo,
                             describe_fn=lambda b, p: '```json\n{"name":"Test","price":1.0}\n```')
    assert r2['name'] == 'Test'
    score += 1; print("\u2705 handles markdown-wrapped JSON response")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def extract_from_image(img_b64: str, schema_cls,
                       describe_fn=None) -> dict:
    prompt = build_extraction_prompt(schema_cls)
    if describe_fn is not None:
        response = describe_fn(img_b64, prompt)
    else:
        import ollama
        resp = ollama.chat(
            model='llava',
            messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
        )
        response = resp['message']['content']
    raw = strip_json_from_response(response)
    return json.loads(raw)
```

**Why return a dict and not a model?** Keeping parse and validate as separate steps makes error handling cleaner: JSONDecodeError = bad syntax (retry the call), ValidationError = bad data (schema mismatch, possibly unretryable). If extraction and validation were combined, you could not distinguish the two failure modes.

</details>